In [1]:
import pandas as pd

df = pd.read_csv("/home/saliherdemk/osu-dataset/encoded.csv", chunksize = 100_000)
df = next(iter(df))

In [2]:
df["count"] = df["tokenized"].str.count(",") + 1

In [3]:
df["count"].max()

np.int64(1911)

In [4]:
df

,beatmap_id,chunk,tokenized,count
0,1000488-0,0,"<beatmap_start>,<hit_object_start>,type_circle...",417
1,1000488-0,1,"<hit_object_start>,type_slider,<start_delta_ti...",507
2,1000488-0,2,"<hit_object_start>,type_slider,<start_delta_ti...",702
3,1000488-0,3,"<hit_object_start>,type_circle,<start_delta_ti...",624
4,1000488-0,4,"<hit_object_start>,type_slider,<start_delta_ti...",754
...,...,...,...,...
999995,2044490-1,11,"<hit_object_start>,type_slider,<start_delta_ti...",572
999996,2044490-1,12,"<hit_object_start>,type_slider,<start_delta_ti...",481
999997,2044490-1,13,"<hit_object_start>,type_slider,<start_delta_ti...",533
999998,2044490-1,14,"<hit_object_start>,type_circle,<start_delta_ti...",598


In [2]:
def last_hit_object_index(tokenized_str):
    tokens = tokenized_str.split(',')
    try:
        last_idx= len(tokens) - 1 - tokens[::-1].index('<hit_object_start>')
        return ','.join(tokens[last_idx:])
    except ValueError:
        return ''

df['last_hit_object'] = df['tokenized'].apply(last_hit_object_index)

In [3]:
df['prev_last_hit_object'] = df.groupby('beatmap_id')['last_hit_object'].shift(1)

In [4]:
df[(df["prev_last_hit_object"].isna()) & (df["chunk"] != 0)]

,beatmap_id,chunk,tokenized,last_hit_object,prev_last_hit_object


In [5]:
prev = df['prev_last_hit_object'].fillna("")

In [6]:
df['tokenized_with_overlap'] = (prev + "," + df['tokenized']).str.strip(",")

In [8]:
grouped = df.groupby("beatmap_id")

In [10]:
samples = {}
for b_id, data in grouped:
    chunks = []
    for _, row in data.sort_values("chunk").iterrows():
        tokens = row["tokenized_with_overlap"].split(",")
        chunks.append(tokens)
    samples[b_id] = chunks
    break
samples

{'1000069-0': [['<beatmap_start>'],
  ['<hit_object_start>',
   'type_slider',
   '<start_delta_time>',
   'dt_0',
   '<end_delta_time>',
   '<start_repeat>',
   'repeat_1',
   '<end_repeat>',
   'sv_1.4',
   '<start_duration>',
   'duration_267',
   '<end_duration>',
   '<hit_object_end>',
   '<hit_object_start>',
   'type_slider',
   '<start_delta_time>',
   'dt_536',
   '<end_delta_time>',
   '<start_repeat>',
   'repeat_1',
   '<end_repeat>',
   'sv_1.4',
   '<start_duration>',
   'duration_267',
   '<end_duration>',
   '<hit_object_end>',
   '<hit_object_start>',
   'type_circle',
   '<start_delta_time>',
   'dt_536',
   '<end_delta_time>',
   '<start_repeat>',
   'repeat_0',
   '<end_repeat>',
   'sv_0.0',
   '<start_duration>',
   'duration_0',
   '<end_duration>',
   '<hit_object_end>',
   '<hit_object_start>',
   'type_slider',
   '<start_delta_time>',
   'dt_536',
   '<end_delta_time>',
   '<start_repeat>',
   'repeat_1',
   '<end_repeat>',
   'sv_1.4',
   '<start_duration>',

In [43]:
a = df[(df["beatmap_id"] == "2044490-1") & (df["chunk"] == 14)]
a["last_hit_object"].iloc[0]

'<hit_object_start>,type_slider,<start_delta_time>,dt_164,<end_delta_time>,<start_repeat>,repeat_1,<end_repeat>,sv_1.3,<start_duration>,duration_327,<end_duration>,<hit_object_end>'

In [44]:
f = df[(df["beatmap_id"] == "2044490-1") & (df["chunk"] == 15)]

f["tokenized_with_overlap"].iloc[0]

'<hit_object_start>,type_slider,<start_delta_time>,dt_164,<end_delta_time>,<start_repeat>,repeat_1,<end_repeat>,sv_1.3,<start_duration>,duration_327,<end_duration>,<hit_object_end>,<hit_object_start>,type_circle,<start_delta_time>,dt_492,<end_delta_time>,<start_repeat>,repeat_0,<end_repeat>,sv_0.0,<start_duration>,duration_0,<end_duration>,<hit_object_end>,<hit_object_start>,type_circle,<start_delta_time>,dt_163,<end_delta_time>,<start_repeat>,repeat_0,<end_repeat>,sv_0.0,<start_duration>,duration_0,<end_duration>,<hit_object_end>,<hit_object_start>,type_circle,<start_delta_time>,dt_82,<end_delta_time>,<start_repeat>,repeat_0,<end_repeat>,sv_0.0,<start_duration>,duration_0,<end_duration>,<hit_object_end>,<hit_object_start>,type_slider,<start_delta_time>,dt_82,<end_delta_time>,<start_repeat>,repeat_1,<end_repeat>,sv_1.3,<start_duration>,duration_163,<end_duration>,<hit_object_end>,<hit_object_start>,type_slider,<start_delta_time>,dt_328,<end_delta_time>,<start_repeat>,repeat_3,<end_repe

In [45]:
f["tokenized"].iloc[0]

'<hit_object_start>,type_circle,<start_delta_time>,dt_492,<end_delta_time>,<start_repeat>,repeat_0,<end_repeat>,sv_0.0,<start_duration>,duration_0,<end_duration>,<hit_object_end>,<hit_object_start>,type_circle,<start_delta_time>,dt_163,<end_delta_time>,<start_repeat>,repeat_0,<end_repeat>,sv_0.0,<start_duration>,duration_0,<end_duration>,<hit_object_end>,<hit_object_start>,type_circle,<start_delta_time>,dt_82,<end_delta_time>,<start_repeat>,repeat_0,<end_repeat>,sv_0.0,<start_duration>,duration_0,<end_duration>,<hit_object_end>,<hit_object_start>,type_slider,<start_delta_time>,dt_82,<end_delta_time>,<start_repeat>,repeat_1,<end_repeat>,sv_1.3,<start_duration>,duration_163,<end_duration>,<hit_object_end>,<hit_object_start>,type_slider,<start_delta_time>,dt_328,<end_delta_time>,<start_repeat>,repeat_3,<end_repeat>,sv_1.3,<start_duration>,duration_81,<end_duration>,<hit_object_end>,<hit_object_start>,type_slider,<start_delta_time>,dt_328,<end_delta_time>,<start_repeat>,repeat_1,<end_repea

In [46]:
df[(df['last_hit_object'] == -1) & (df['count'] != 2)]

,beatmap_id,chunk,tokenized,count,last_hit_object,prev_last_hit_object,tokenized_with_overlap


# DataLoader

In [13]:
import torch
from torch.utils.data import Dataset
import numpy as np
import os

class BeatmapDataset(Dataset):
    def __init__(self, df, mel_folder):
        self.mel_folder = mel_folder
        self.samples = self.get_samples(df) 

    def get_samples(self, df):
        df['last_hit_object'] = df['tokenized'].apply(self.last_hit_object)
        df['prev_last_hit_object'] = df.groupby('beatmap_id')['last_hit_object'].shift(1)
        prev = df["prev_last_hit_object"].fillna("")
        df["tokenized_with_overlap"] = (prev + "," + df["tokenized"]).str.strip(",")
        samples = []
        for _, row in df.iterrows():
            mel_path = os.path.join(self.mel_folder, f"{row['beatmap_id'].split("-")[0]}_{row['chunk']}.npy")
            samples.append({
                "mel_path": mel_path,
                "tokenized": row["tokenized_with_overlap"]
            })
        samples = {}
        for b_id, data in df.groupby("beatmap_id"):
            chunks = []
            for _, row in data.sort_values("chunk").iterrows():
                tokens = row["tokenized_with_overlap"].split(",")
                chunks.append(tokens)
            samples[b_id] = chunks
        return samples


    def last_hit_object(self, tokenized_str):
        tokens = tokenized_str.split(',')
        try:
            last_idx= len(tokens) - 1 - tokens[::-1].index('<hit_object_start>')
            return ','.join(tokens[last_idx:])
        except ValueError:
            return ''

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        beatmap_id = list(self.samples.keys())[idx]
        chunks = self.samples[beatmap_id]
    
        mel_list = []
        for chunk_idx in range(len(chunks)):
            mel_path = os.path.join(self.mel_folder, f"{beatmap_id.split('-')[0]}_{chunk_idx}.npy")
            mel = np.load(mel_path)
            mel_list.append(mel)
    
        return mel_list, chunks



In [14]:
from torch.utils.data import DataLoader

dataset = BeatmapDataset(df, mel_folder="/home/saliherdemk/osu-dataset/mels/")

dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2
)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [18]:
for batch in dataloader:
    mel, c = batch
    print(len(mel))
    print(len(c))
    break


21
21
